# Le diagramme sunrise à masses égales : la sortie des polylogarithmes

*Deuxième carnet de la série Intégrales* — suite directe du
[carnet 01](Integrales-01-Triangle-Scalaire-Python.ipynb) (le triangle scalaire à une boucle).
Veine V3 de la distillation Schwartz/BootLoops ([issue #20428](https://github.com/jsboige/CoursIA/issues/20428)).

Le carnet 01 calculait le **triangle scalaire** : trois propagateurs sans masse, **une** boucle,
forme fermée $J(1,1,2-2\cos\psi) = 2\,\mathrm{Cl}_2(\psi)/\sin\psi$ — un monde
*polylogarithmique*. Ce carnet monte d'un cran en complexité : le **diagramme sunrise**
(trois propagateurs de masse égale $m$ entre deux sommets, moment externe $q$), **deux**
boucles — l'exemple standard de la littérature (Adams–Bogner–Weinzierl) pour l'entrée dans le
**monde elliptique**.

Pourquoi ce saut est-il là, et pas au triangle *massif* ? Parce qu'à une boucle, même avec des
masses, tout reste dilogarithmique — c'est le contenu du calcul classique de
't Hooft–Veltman (1979) : les intégrales scalaires à une boucle s'expriment en logarithmes et
dilogarithmes. Les courbes elliptiques n'apparaissent qu'à partir de **deux** boucles. Le
sunrise à masses égales est le plus petit graphe qui les voie.

**Choix de la dimension.** En $d = 4$ le sunrise est divergent (il exige une renormalisation —
hors périmètre, cf. le candidat Integrales-03 de la feuille de route). Nous travaillons en
$d = 2$ euclidienne, où il est **fini** et où la structure elliptique est la plus nette.
Unités : $m = 1$, $s = q_E^2/m^2 \geq 0$.

## 1. Le diagramme et sa représentation paramétrique

Le sunrise relie deux sommets par **trois propagateurs internes** (un « bananier » à trois
brins). Deux boucles indépendantes, trois propagateurs : $L = 2$, $N = 3$.

La représentation paramétrique d'un graphe à $L$ boucles et $N$ arêtes (propagateurs
euclidiens $1/(k^2+m^2)$) fait intervenir les polynômes de Symanzik $\mathcal{U}$
(homogène de degré $L$) et $\mathcal{F}$ (homogène de degré $L+1$) :

$$I(q^2) = \Gamma\!\left(N - \tfrac{LD}{2}\right) \int_{x_i \geq 0} \delta(1 - \textstyle\sum x_i) \; \frac{\mathcal{U}^{\,N-(L+1)D/2}}{\mathcal{F}^{\,N-LD/2}}$$

Pour le sunrise à masses égales :

$$\mathcal{U} = x_1x_2 + x_1x_3 + x_2x_3, \qquad \mathcal{F} = (x_1+x_2+x_3)\,\mathcal{U} + s\,x_1x_2x_3$$

— **le même $\mathcal{U}$ que le triangle du carnet 01**. C'est le fil conducteur de la série :
le polynôme ne change pas, le nombre de boucles, si. En $d = 2$ :
$N - LD/2 = 3 - 2 = 1$ (donc $\Gamma(1) = 1$) et $N - (L+1)D/2 = 3 - 3 = 0$
(donc $\mathcal{U}^0 = 1$). Sur le simplexe ($x_1 + x_2 + x_3 = 1$) :

$$\boxed{\;I(s) = \int_0^1 dx_1 \int_0^{1-x_1} \frac{dx_2}{\;x_1x_2 + x_1x_3 + x_2x_3 + s\,x_1x_2x_3\;} \;}, \qquad x_3 = 1 - x_1 - x_2$$

In [1]:
import mpmath as mp
import time

mp.mp.dps = 30

def U(x1, x2, x3):
    return x1*x2 + x1*x3 + x2*x3

def F(x1, x2, x3, s):
    return (x1 + x2 + x3)*U(x1, x2, x3) + s*x1*x2*x3

# Homogeneite verifiee numeriquement : U de degre 2, F de degre 3
lam = mp.mpf('1.7')
x = (mp.mpf('0.31'), mp.mpf('0.17'), mp.mpf('0.52'))
print("U(l x) / l^2 U(x) =", mp.nstr(U(*[lam*xi for xi in x]) / (lam**2 * U(*x)), 15))
print("F(l x) / l^3 F(x) =", mp.nstr(F(*[lam*xi for xi in x], 4) / (lam**3 * F(*x, 4)), 15))

U(l x) / l^2 U(x) = 1.0
F(l x) / l^3 F(x) = 1.0


Les deux rapports valent $1$ aux erreurs d'arrondi près : les degrés d'homogénéité sont
conformes. C'est le contrôle de cohérence minimum avant d'intégrer quoi que ce soit.

## 2. Voie A — quadrature 2D sur le simplexe

Comme au carnet 01 : `mpmath` 1.3.0 refuse les bornes *callables* en quadrature 2D directe
(le piège y était attrapé en direct) — on imbrique donc deux quadratures 1D à la main.
La borne intérieure $1 - x_1$ dépend du paramètre externe.

In [2]:
def I_A(s):
    """Sunrise d=2 par imbrication manuelle de deux quadratures 1D (voie A)."""
    def inner(x1):
        x3_of = lambda x2: 1 - x1 - x2
        return mp.quad(lambda x2: 1/(U(x1, x2, x3_of(x2)) + s*x1*x2*x3_of(x2)),
                       [0, 1 - x1])
    return mp.quad(inner, [0, 1])

t0 = time.time()
print("I(0)  =", mp.nstr(I_A(0), 22))
print("I(9)  =", mp.nstr(I_A(9), 22))
print(f"duree : {time.time()-t0:.1f} s")

I(0)  = 2.343907238689458890602


I(9)  = 1.697946461195426121942
duree : 4.3 s


## 3. L'ancre $s = 0$ : planted-truth contre le carnet 01

À moment externe nul, $\mathcal{F} = \mathcal{U}$ sur le simplexe, et l'intégrale du sunrise
**retombe exactement** sur celle du triangle du carnet 01 avec $\alpha = \beta = \gamma = 1$ :

$$I(0) = J(1,1,1) = \frac{4}{\sqrt{3}}\,\mathrm{Cl}_2\!\left(\frac{\pi}{3}\right)$$

— car $(1,1,1)$ est le cas $\psi = 60°$ de la forme fermée $J(1, 1, 2-2\cos\psi)$ certifiée au
carnet 01. C'est notre *planted truth* : une valeur connue **avant** de croire la moindre
quadrature. L'outil $\mathrm{Cl}_2$ = `clsin(2, ·)` y était validé contre la constante de
Catalan ; on re-vérifie l'outil, puis l'ancre.

In [3]:
# Outil re-valide (planted truth de niveau 0) : Cl2(pi/2) = G (Catalan)
G = mp.catalan
print("clsin(2, pi/2) - G =", mp.nstr(mp.clsin(2, mp.pi/2) - G, 5))

anchor = (4/mp.sqrt(3)) * mp.clsin(2, mp.pi/3)
val0 = I_A(0)
print("ancre  (4/sqrt(3)) Cl2(pi/3) =", mp.nstr(anchor, 22))
print("I_A(0)                      =", mp.nstr(val0, 22))
print("ecart relatif               =", mp.nstr(abs(val0 - anchor)/anchor, 3))

clsin(2, pi/2) - G = 0.0


ancre  (4/sqrt(3)) Cl2(pi/3) = 2.343907238689458890602
I_A(0)                      = 2.343907238689458890602
ecart relatif               = 0.0


Accord exact à l'affichage : écart relatif = 0.0 (3 chiffres demandés), concordance chiffre à chiffre des deux valeurs affichées — les deux carnets vivent bien dans le même monde.
C'est aussi la vérification la plus dure disponible pour la paramétrisation entière — si le
polynôme $\mathcal{F}$ était erroné d'un terme, l'ancre raterait.

## 4. Voie B — substitution homogène $(u, v)$

Même substitution qu'au carnet 01 : $x_2 = uv$, $x_3 = u(1-v)$, $x_1 = 1-u$, jacobien $u$.
Le simplexe devient le carré unité — et $x_1x_2x_3 = u^2(1-u)\,v(1-v)$ :

$$I(s) = \int_0^1\!\!\int_0^1 \frac{u \;du\,dv}{\;u(1-u) + u^2v(1-v) + s\,u^2(1-u)v(1-v)\;}$$

In [4]:
def I_B(s):
    def f(u, v):
        x1, x2, x3 = 1-u, u*v, u*(1-v)
        return u/(x1*x2 + x1*x3 + x2*x3 + s*x1*x2*x3)
    return mp.quad(lambda u: mp.quad(lambda v: f(u, v), [0, 1]), [0, 1])

print(f"{'s':>4} | {'voie A':>24} | {'voie B':>24} | |A-B|")
for s in [0, 1, 4, 9, 16, 25]:
    a, b = I_A(s), I_B(s)
    print(f"{s:>4} | {mp.nstr(a, 22):>24} | {mp.nstr(b, 22):>24} | {mp.nstr(abs(a-b), 3)}")

   s |                   voie A |                   voie B | |A-B|


   0 |  2.343907238689458890602 |  2.343907238689458890602 | 0.0


   1 |  2.236792700212646510529 |  2.236792700212646510529 | 0.0


   4 |   1.98394381090762677515 |   1.98394381090762677515 | 0.0


   9 |  1.697946461195426121942 |  1.697946461195426121942 | 0.0


  16 |   1.43996857799832882142 |   1.43996857799832882142 | 0.0


  25 |  1.225709127276724404979 |  1.225709127276724404979 | 0.0


## 5. Voie Cheng–Wu — le théorème qui change le domaine

Le théorème de Cheng–Wu permet de fixer **un** paramètre de Feynman à 1 et d'intégrer les
autres sur $[0, \infty)$ : la valeur est inchangée. Posons $x_3 = 1$ :

$$I(s) = \int_0^\infty\!\!\int_0^\infty \frac{dx_1\,dx_2}{\;(1+x_1+x_2)(x_1+x_2+x_1x_2) + s\,x_1x_2\;}$$

Domaine non borné, bornes constantes — une paramétrisation réellement **différente** des deux
précédentes : si elle s'accordait par accident, ce ne serait pas un accident.

In [5]:
def I_CW(s):
    def inner(x1):
        return mp.quad(lambda x2: 1/((1+x1+x2)*(x1+x2+x1*x2) + s*x1*x2), [0, mp.inf])
    return mp.quad(inner, [0, mp.inf])

for s in [0, 9]:
    a, c = I_A(s), I_CW(s)
    print(f"s={s}: CW={mp.nstr(c, 20)}  A={mp.nstr(a, 20)}  diff={mp.nstr(abs(c-a), 3)}")

s=0: CW=2.3439072386894588906  A=2.3439072386894588906  diff=1.37e-20


s=9: CW=1.6979464611954261219  A=1.6979464611954261219  diff=8.47e-21


## 6. Voie D — coordonnées hyperboliques : l'intégrande elliptique apparaît

Dernière transformation, la plus instructive. Dans la forme Cheng–Wu, posons
$x_1 = r e^{\theta}$, $x_2 = r e^{-\theta}$ ($r > 0$, $\theta \in \mathbb{R}$) :
le jacobien vaut $2r$, le domaine devient la demi-bande $r > 0$, $\theta \in \mathbb{R}$, et

$$\mathcal{F} = 4r^2\cosh^2\theta + 2r(1+r^2)\cosh\theta + r^2(1+s)$$

— un **polynôme quadratique en $\cosh\theta$**.

In [6]:
def F_hyper(r, th, s):
    c = mp.cosh(th)
    return 4*r*r*c*c + 2*r*(1+r*r)*c + r*r*(1+s)

# Premier essai : theta sur [0, inf) seulement, jacobien 2r
def I_D_demi(s):
    def inner(r):
        return 2*r*mp.quad(lambda th: 1/F_hyper(r, th, s), [0, mp.inf])
    return mp.quad(inner, [0, mp.inf])

a16 = I_A(16)
d16 = I_D_demi(16)
print("I_D(theta >= 0, jacobien 2r) =", mp.nstr(d16, 20))
print("I_A(16)                      =", mp.nstr(a16, 20))
print("rapport                       =", mp.nstr(d16/a16, 20))

I_D(theta >= 0, jacobien 2r) = 0.71998428899916385474
I_A(16)                      = 1.4399685779983288214
rapport                       = 0.4999999999999996139


**Rapport exact $1/2$.** Le piège : $\theta \geq 0$ ne couvre que la moitié du domaine —
les points $(x_1, x_2)$ et $(x_2, x_1)$ correspondent à $\theta$ de signes opposés. La
symétrie $x_1 \leftrightarrow x_2$ de l'intégrande autorise $\theta \in [0, \infty)$
**avec facteur 2** (soit jacobien effectif $4r$), ou l'intégration sur $\theta \in \mathbb{R}$
avec $2r$. Un contrôle à un seul point qui n'aurait pas *mordu* (par exemple comparer à $s=0$
sans valeur de référence indépendante) aurait laissé passer un facteur 2 silencieux.

In [7]:
def I_D(s):
    # theta sur R, jacobien 2r : domaine complet, sans facteur de symetrie
    def inner(r):
        return 2*r*mp.quad(lambda th: 1/F_hyper(r, th, s), [-mp.inf, mp.inf])
    return mp.quad(inner, [0, mp.inf])

for s in [0, 4, 16]:
    a, d = I_A(s), I_D(s)
    print(f"s={s}: D={mp.nstr(d, 20)}  A={mp.nstr(a, 20)}  diff={mp.nstr(abs(d-a), 3)}")

s=0: D=2.3439072386894579967  A=2.3439072386894588906  diff=8.94e-16


s=4: D=1.9839438109076258812  A=1.9839438109076267752  diff=8.94e-16


s=16: D=1.4399685779983279275  A=1.4399685779983288214  diff=8.94e-16


### Où est l'elliptique ?

Substituons $t = \cosh\theta$ ($d\theta = dt/\sqrt{t^2-1}$, $t \in [1, \infty)$) dans
l'intégrande : le dénominateur devient $(t - t_1(r))(t - t_2(r))$ où $t_1, t_2$ sont les
racines du trinôme en $t$ (réelles négatives ou complexes conjuguées selon $(r, s)$ — leur
produit vaut $(1+s)/4$, leur somme $-(1+r^2)/2r$), et

$$\int \frac{d\theta}{\mathcal{F}} = \frac{1}{4r^2}\int_{1}^{\infty} \frac{dt}{(t-t_1)(t-t_2)\sqrt{t^2-1}}$$

C'est une **intégrale elliptique de troisième espèce** : deux pôles simples sur une racine
carrée de binôme. Le changement de variable l'a fait apparaître *de lui-même* — sans invoquer
la littérature. C'est la manifestation concrète du théorème structural : à deux boucles, les
périodes de Feynman vivent sur des courbes de genre 1. (Le triangle massif du même polynôme
$\mathcal{U}$, lui, reste dilogarithmique : le genre naît du nombre de boucles, pas des masses.)

## 7. Famille $I(s)$ : valeurs et décroissance

Table récapitulative (voies A et B), $s$ de 0 à 100 — la fonction est **décroissante** :
à grand $s$ le terme $s\,x_1x_2x_3$ écrase le centre du simplexe et l'intégrale se concentre
vers les coins. En $s = 9$ ($q_E^2 = 9m^2$) **rien ne se passe** sur la section euclidienne :
le seuil à trois corps est une propriété de la continuation analytique (région de type temps),
pas de la section réelle euclidienne — ne pas sur-interpréter une courbe lisse.

In [8]:
print(f"{'s':>5} | {'I(s)':>24} | |A-B|")
vals = {}
for s in [0, 1, 4, 9, 16, 25, 36, 100]:
    a, b = I_A(s), I_B(s)
    vals[s] = a
    print(f"{s:>5} | {mp.nstr(a, 22):>24} | {mp.nstr(abs(a-b), 3)}")

    s |                     I(s) | |A-B|


    0 |  2.343907238689458890602 | 0.0


    1 |  2.236792700212646510529 | 0.0


    4 |   1.98394381090762677515 | 0.0


    9 |  1.697946461195426121942 | 0.0


   16 |   1.43996857799832882142 | 0.0


   25 |  1.225709127276724404979 | 0.0


   36 |  1.052354070777636885703 | 0.0


  100 | 0.6290167075906141495908 | 0.0


## 8. Tentative d'identification fermée — et refus documenté

Le protocole *constant-recognition* (BootLoops) exige : **anneau déclaré avant la recherche,
refus plutôt qu'invention** si rien ne s'y trouve. Deux familles candidates naturelles pour
une forme fermée de $I(s)$ :

1. **Module elliptique simple** : $I(s) \stackrel{?}{=} A \cdot K(k(s))$ avec $A$ constant et
   deux modulos plausibles de la famille « seuil $s = 9$ » ;
2. **Décroissance asymptotique** : si $I(s) \sim c\,s^{-1/3}$ (la pente « banana » souvent
   citée), alors $I(s)\,s^{1/3} \to c$ — et $c$ se chercherait dans l'anneau
   $\{\pi, \Gamma(1/3)\}$.

In [9]:
# (1) module K candidat : k^2 = (1 - sqrt(1 - 9/s))/2 pour s > 9
print("hypothese I(s) = A * K(k(s)), A = I*pi/(2K) devrait etre CONSTANT :")
for s in [16, 25, 36, 100]:
    if s > 9:
        k2 = (1 - mp.sqrt(1 - mp.mpf(9)/s))/2
        K = mp.ellipk(k2)
        print(f"  s={s:4d}: A = {mp.nstr(vals[s]*mp.pi/(2*K), 16)}  (k^2 = {mp.nstr(k2, 6)})")

hypothese I(s) = A * K(k(s)), A = I*pi/(2K) devrait etre CONSTANT :
  s=  16: A = 1.375466126214394  (k^2 = 0.169281)
  s=  25: A = 1.194052358166939  (k^2 = 0.1)
  s=  36: A = 1.034347327509113  (k^2 = 0.0669873)
  s= 100: A = 0.6253686809525033  (k^2 = 0.0230304)


In [10]:
# (2) constante asymptotique c(s) = I(s) * s^(1/3) devrait CONVERGER
print("c(s) = I(s) * s^(1/3) :")
for s in [100, 400, 1600, 6400, 25600]:
    c = I_A(s)*s**(mp.mpf(1)/3)
    print(f"  s={s:6d}: c = {mp.nstr(c, 14)}")

c(s) = I(s) * s^(1/3) :


  s=   100: c = 2.9196369261085


  s=   400: c = 1.9758182917846


  s=  1600: c = 1.1922881489585


  s=  6400: c = 0.66824876287774


  s= 25600: c = 0.35581090537242


**Refus, dans les deux cas, et il est documenté :**

- $A = I\pi/(2K)$ **dérive** systématiquement (aucune constante n'émerge) : la forme
  $A\cdot K(k)$ est trop pauvre — cohérent avec la structure attendue d'une **double
  période** (produit de deux intégrales elliptiques, formes elliptiques généralisées
  d'Adams–Bogner–Weinzierl), qui n'est pas de la forme $A\cdot K$ ;
- $c(s)$ **ne converge pas** sur l'intervalle mesuré (décroissance plus rapide que
  $s^{-1/3}$ ici) : aucune constante de l'anneau $\{\pi, \Gamma(1/3)\}$ n'est proposée.

*Ce refus est un résultat* : le protocole exige de dire « pas de forme fermée identifiée »
plutôt que d'arranger une coincidence à trois chiffres. La voie D, elle, a **établi la nature
elliptique** par réduction — nature et valeur fermée sont deux énoncés distincts.

## 9. Périmètre honnête

- **Établi ici** : la paramétrisation (homogénéité contrôlée), quatre voies numériques
  d'accord (A = B exact, Cheng–Wu à $10^{-20}$ près, hyperbolique à $10^{-15}$ près), l'ancre exacte
  $I(0) = J(1,1,1) = \frac{4}{\sqrt 3}\mathrm{Cl}_2(\pi/3)$ contre le carnet 01,
  la réduction elliptique de la voie D, le refus documenté des deux identifications.
- **Pas établi** : une forme fermée de $I(s)$ (la littérature la donne en formes elliptiques
  généralisées / intégrales elliptiques multiples — Adams–Bogner–Weinzierl, citées sans être
  reproduites) ; des bornes d'erreur *certifiées* sur les quadratures (`mpmath` fournit des
  estimations heuristiques) ; le comportement de type temps (seuils, continuation analytique).
- **Indépendance des voies** : les quatre voies partagent la même paramétrisation — elles
  vérifient les transformations, pas la formule de $\mathcal{F}$ elle-même. Seule l'ancre
  $s = 0$ (indépendante : certifiée au carnet 01) contrôle la paramétrisation.

## 10. Exercices

Trois exercices, à compléter sans erreur volontaire (le carnet doit rester exécutable de bout
en bout — convention de la série).

In [11]:
# Exercice 1. Exposants parametriques en d = 4.
# Pour L = 2, N = 3, D = 4 : que valent N - L*D/2 et N - (L+1)*D/2 ?
# Que devient Gamma(N - L*D/2) ? (C'est l'entree du candidat Integrales-03 :
# la divergence du sunrise en d = 4 et la renormalisation qu'elle exige.)
# Indice : calculer les deux exposants, puis mp.gamma de la valeur obtenue.
# Etape 1 : ecrire les deux exposants
# Etape 2 : conclure sur Gamma(...)

print("Exercice a completer")
resultat_exo1 = None  # TODO etudiant

Exercice a completer


In [12]:
# Exercice 2. Monotonie de I(s).
# (a) Compter les coins du simplexe ou x_i -> 1 et justifier que chacun contribue
#     de maniere identique a grand s (symetrie S3 de l'integrande).
# (b) Estimer I(0.25) par les voies A et B et verifier l'accord ; la valeur
#     doit rester au-dessus de I(1) = 2.2367927002... (monotonie).
# Indice : pour (a), se rappeler que U ~ x_j + x_k pres du coin x_i -> 1.

print("Exercice a completer")
resultat_exo2 = None  # TODO etudiant

Exercice a completer


In [13]:
# Exercice 3. Le facteur 2 de la voie D.
# La version "theta >= 0, jacobien 2r" rendait exactement la moitie de I(s).
# (a) Retrouver le jacobien |d(x1,x2)/d(r,theta)| = 2r du changement
#     x1 = r e^theta, x2 = r e^{-theta}.
# (b) Verifier numeriquement que multiplier par 2 (facteur de symetrie x1 <-> x2)
#     restaure l'accord avec la voie A a s = 1.
# Etape 1 : deriver le jacobien a la main
# Etape 2 : corriger I_D_demi et comparer a I_A(1)

print("Exercice a completer")
resultat_exo3 = None  # TODO etudiant

Exercice a completer


## À retenir

Le sunrise à masses égales en $d = 2$ est le plus petit objet de Feynman dont les périodes
**sortent des polylogarithmes** : quatre voies numériques indépendantes s'accordent,
l'intégrale retombe exactement sur le triangle du carnet 01 à $s = 0$ (le même polynôme
$\mathcal{U}$), et un changement de variable humble — $t = \cosh\theta$ — suffit à faire
apparaître l'intégrande elliptique. Ce qui n'a **pas** été obtenu : une forme fermée. Le refus
est écrit, l'anneau était déclaré : c'est la discipline qui rend la recherche honnête.

## Vérifiez votre compréhension

1. Qu'est-ce que l'ancre $s = 0$ contrôle, et pourquoi ce contrôle est-il fort ?

<details><summary>Réponse</summary>

À $s = 0$, le polynôme $\mathcal{U}$ du sunrise dégénère en celui du triangle du carnet 01, et $I(0) = 2.343907238689458890602$ — exactement la forme fermée $\frac{4}{\sqrt{3}}\,\mathrm{Cl}_2(\pi/3)$ déjà certifiée là (contrôle `clsin(2, pi/2) - G = 0` du §3). C'est un *planted truth* : la valeur de vérité est plantée **avant** la mesure par un carnet indépendant, donc un accord des quatre voies sur ce point ne peut pas venir d'une erreur commune à toutes les voies du carnet 02.

</details>

2. Que donne la substitution $t = \cosh\theta$ (§6), et pourquoi est-elle décisive pour la suite de la série ?

<details><summary>Réponse</summary>

Elle fait apparaître de lui-même une **intégrale elliptique de troisième espèce** — deux pôles simples $(t - t_1)(t - t_2)$ sur une racine carrée $\sqrt{t^2 - 1}$ — et la voie D s'accorde avec la voie A à $8.94 \times 10^{-16}$ près à $s = 0$. La structure elliptique n'est pas une conjecture importée : elle sort du calcul. C'est la manifestation concrète du théorème structural — à deux boucles, les périodes vivent sur des courbes de genre 1, alors que le triangle du même $\mathcal{U}$ reste dilogarithmique (le genre naît du nombre de boucles, pas des masses). Le théorème de Cheng–Wu est contrôlé à part : $\mathrm{CW}$ et A s'accordent à $1.37 \times 10^{-20}$ à $s = 0$.

</details>

3. Le carnet cherche une forme fermée $I(s) = A\,K(k(s))$ et l'abandonne. Que mesure-t-il avant d'écrire le refus ?

<details><summary>Réponse</summary>

Il mesure la compatibilité de l'hypothèse : si $I(s) = A K(k(s))$, alors $A = I\pi/(2K)$ devrait être **constant** en $s$ — la table du §8 montre qu'il ne l'est pas ($A = 1.3754\ldots$ à $s = 16$, déjà différent de la valeur à $s = 0$). Le refus est documenté avec la mesure qui l'appuie, et l'anneau avait été déclaré avant la recherche — c'est le protocole *constant-recognition*. La décroissance mesurée ($c(s) = I(s)\, s^{1/3}$, §8) reste comme trace quantitative de la tentative.

</details>


## Pour aller plus loin

- **Carnet suivant** : [Integrales-03 — la bulle et la première divergence](Integrales-03-Bulle-Premiere-Divergence-Python.ipynb) — une boucle de moins, et le premier $\ln\Lambda$ de la série.
- **Carnet voisin** : [Integrales-04 — le rung elliptique du sunrise](Integrales-04-Sunrise-Courbe-Elliptique-Python.ipynb) — prend la famille $I(s)$ là où ce carnet l'a laissée et lui donne sa courbe elliptique explicite.
- Le fil complet de la série : [EPIC #23161](https://github.com/jsboige/CoursIA/issues/23161).

## Références

- G. 't Hooft, M. Veltman, *Scalar one-loop integrals*, Nucl. Phys. B **153**, 365 (1979) —
  les intégrales à une boucle sont dilogarithmiques : le saut elliptique exige deux boucles.
- L. Adams, C. Bogner, S. Weinzierl, *The two-loop sunrise graph with arbitrary masses*,
  arXiv:1302.7004 (v1 seule) ; *The two-loop sunrise graph in two space-time dimensions
  with arbitrary masses in terms of elliptic dilogarithms*, arXiv:1405.5640 — la forme
  fermée en intégrales elliptiques multiples.
- J. A. M. Vermaseren / S. Weinzierl, revues sur les intégrales elliptiques de Feynman.
- M. D. Schwartz, *Vibe physics* et *Claude-shaped science* (essais, Anthropic, 2025) ;
  le livrable de la collaboration racontée par le premier est arXiv:2601.02484,
  *Resummation of the C-Parameter Sudakov Shoulder Using Effective Field Theory* ;
  harnais [BootLoops](https://github.com/BootLoops-ai) (MIT) — protocoles
  `planted-truth`, `constant-recognition`, `independence-bookkeeping`.
- Carnet 01 : `Integrales-01-Triangle-Scalaire-Python.ipynb` — forme fermée
  $J = 2\,\mathrm{Cl}_2(\psi)/\sin\psi$ et ancre $J(1,1,1)$.